# Step 10 — Bloom prediction

Steps 05 and 09 forecast the chlorophyll value, and both underestimate blooms: a forecast that is right on
average leans towards the usual value. This step asks a different question: **how likely is a bloom?**

* A **bloom** is a pixel-day whose observed chlorophyll is at least 10^0.2 ≈ 1.6 × the usual value for its
  month (`config.BLOOM_ANOMALY_LOG10`), the "high-chlorophyll days" of steps 05 and 09.
* Forecasts are **probabilities**, scored with the **Brier score** (mean squared difference between the
  probability and what happened, 1 or 0) and the **Brier skill score**, its % improvement over a reference.
* Simple rules to beat: the **usual bloom frequency** for the pixel and month, and the **anomaly rule**:
  how often a bloom followed when the water was this far from normal.
* Candidates: a **classifier** trained on the 18 inputs of the model in use; the same with two inputs
  describing **the whole area over the past 30 days** (blooms vary greatly from year to year); and the
  **model in use** (step 09) turned into a probability.
* Same years and rules as before: train 2017–2023, choose on 2024, test once on 2025; 95% ranges from
  resampling whole days. The yes/no warning threshold is chosen on 2024.

In [1]:
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from chlforecast import blooms, config, samples
from chlforecast import features as F

TRAIN_SAMPLE_FRACTION = 0.3        # screening on 2024
FINAL_SAMPLE_FRACTION = 0.6        # final fit on 2017–2024, as in steps 05 and 09
HISTORY_START = "2016-12-01"
ANOMALY_BINS = np.round(np.arange(-0.6, 0.81, 0.1), 2)   # bands of the latest anomaly, for the anomaly rule
COAST_KM = 5
SEASONS = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
           6: "Summer", 7: "Summer", 8: "Summer", 9: "Autumn", 10: "Autumn", 11: "Autumn"}
horizons = list(config.FORECAST_HORIZONS)

TREE_PARAMS = dict(learning_rate=0.05, max_iter=150, max_leaf_nodes=31, min_samples_leaf=100,
                   l2_regularization=1.0, early_stopping=False, random_state=42)   # as steps 05 and 09

in_use = joblib.load(config.MODEL_PATH)   # the model in use (step 09): its inputs, climatology and models
FEATURES = in_use["feature_names"]
print(f"Model in use: {config.MODEL_PATH.name}, {len(FEATURES)} inputs")

Model in use: chl_forecast_step09.joblib, 18 inputs


In [2]:
observations = xr.open_dataset(config.PROCESSED / "malta_daily_observations.nc").sel(time=slice(HISTORY_START, None))
dates = pd.DatetimeIndex(observations.time.values)
latitude, longitude = observations.latitude.values, observations.longitude.values
GRID = ("time", "latitude", "longitude")
chl = F.quality_control(observations["chl"].transpose(*GRID).values, minimum=config.MIN_PLAUSIBLE_CHL)
sst = observations["sst_c"].transpose(*GRID).values

mask = in_use["model_mask"].values == 1
monthly_log_clim = in_use["monthly_log_climatology"].values
coast_km = in_use["coast_distance_km"].values
inputs = F.model_inputs(chl, sst, dates, mask, monthly_log_clim, in_use["monthly_sst_climatology"].values,
                        coast_km, latitude, longitude)
inputs.update(blooms.area_state_inputs(chl, dates, mask, monthly_log_clim))
FEATURE_SETS = {"18 inputs": FEATURES, "18 inputs + area state": FEATURES + ["area_anomaly_30d", "area_bloom_share_30d"]}

targets, target_clim = {}, {}
for h in horizons:
    target = np.full(chl.shape, np.nan, dtype="float32")
    target[:-h] = chl[h:]
    targets[h] = target
    target_clim[h] = F.climatology_for_dates(monthly_log_clim, dates + pd.Timedelta(days=h))

last_target = dates + pd.Timedelta(days=max(horizons))
periods = {name: (dates >= pd.Timestamp(a)) & (last_target <= pd.Timestamp(b))
           for name, (a, b) in config.SPLIT_PERIODS.items()}
periods["train+validation"] = periods["train"] | periods["validation"]
n_pixels = mask.size

# How common are blooms?
counts = []
for name in ["train", "validation", "test"]:
    for h in horizons:
        valid = np.isfinite(targets[h]) & (targets[h] > 0) & mask[None] & periods[name][:, None, None]
        b = np.log10(targets[h][valid]) - target_clim[h][valid] >= config.BLOOM_ANOMALY_LOG10
        counts.append({"Period": name, "Horizon": f"+{h}d", "Observed pixel-days": int(valid.sum()),
                       "Bloom pixel-days": int(b.sum()), "Bloom share (%)": 100 * b.mean()})
display(pd.DataFrame(counts).set_index(["Period", "Horizon"]).style.format(
    {"Observed pixel-days": "{:,.0f}", "Bloom pixel-days": "{:,.0f}", "Bloom share (%)": "{:.2f}"}))

In [3]:
def rows(period, h, fraction, seed=0):
    """Rows (issue day, pixel) with an observed target, plus what the rules and scores need."""
    r = samples.rows_from_arrays(inputs, targets[h], periods[period], mask, fraction, seed)
    r["clim_log"] = target_clim[h][r["t"], r["i"], r["j"]].astype("float64")
    r["bloom"] = blooms.is_bloom(r["observed"], r["clim_log"])
    r["anomaly"] = np.asarray(inputs["chl_anomaly_log"])[r["t"], r["i"], r["j"]].astype("float64")
    r["month"] = (dates[r["t"]] + pd.Timedelta(days=h)).month.to_numpy()
    r["pixel"] = r["i"] * mask.shape[1] + r["j"]
    return r


def fit_rules(train):
    return {"base": blooms.base_rate_table(train["bloom"], train["month"], train["pixel"], n_pixels),
            "anomaly": blooms.anomaly_rule_table(train["bloom"], train["anomaly"], ANOMALY_BINS)}


def apply_rules(rules, r):
    return {"Usual frequency": rules["base"][r["month"] - 1, r["pixel"]],
            "Anomaly rule": blooms.anomaly_rule(rules["anomaly"], r["anomaly"], ANOMALY_BINS)}


def calibrate_regression(regressor, train):
    """Turn a chlorophyll forecast into a bloom probability: a logistic curve of the
    forecast's anomaly (log10 forecast − usual value), fitted on the training rows."""
    anomaly = regressor.predict(train["X"][FEATURES]) - train["clim_log"]
    return LogisticRegression().fit(anomaly[:, None], train["bloom"])


def regression_probability(regressor, curve, r):
    anomaly = regressor.predict(r["X"][FEATURES]) - r["clim_log"]
    return curve.predict_proba(anomaly[:, None])[:, 1]

In [4]:
# Screening on 2024. The regression candidate is refitted on 2017–2023 here, because
# the model in use was trained on 2017–2024 and has seen 2024.
screen_rows, thresholds, screen_probs = [], {}, {}
for h in horizons:
    started = perf_counter()
    train = rows("train", h, TRAIN_SAMPLE_FRACTION)
    valid = rows("validation", h, 1.0)
    rules = fit_rules(train)
    probs = apply_rules(rules, valid)
    for name, feature_set in [("Classifier", "18 inputs"), ("Classifier + area state", "18 inputs + area state")]:
        names = FEATURE_SETS[feature_set]
        classifier = HistGradientBoostingClassifier(loss="log_loss", **TREE_PARAMS).fit(train["X"][names], train["bloom"])
        probs[name] = classifier.predict_proba(valid["X"][names])[:, 1]
    regressor = HistGradientBoostingRegressor(loss="squared_error", **TREE_PARAMS).fit(train["X"][FEATURES], np.log10(train["observed"]))
    probs["Model in use, as a probability"] = regression_probability(regressor, calibrate_regression(regressor, train), valid)
    for name, p in probs.items():
        skill, low, high = blooms.brier_skill_interval(valid["bloom"], p, probs["Usual frequency"], valid["t"])
        vs_rule = blooms.brier_skill_interval(valid["bloom"], p, probs["Anomaly rule"], valid["t"])
        screen_rows.append({"Horizon": f"+{h}d", "Method": name, "Brier score": blooms.brier(p, valid["bloom"]),
                            "Skill vs usual frequency (%)": skill, "95% low": low, "95% high": high,
                            "Skill vs anomaly rule (%)": vs_rule[0], "AUC": roc_auc_score(valid["bloom"], p)})
        thresholds[(h, name)] = blooms.best_threshold(p, valid["bloom"])
    screen_probs[h] = (valid["bloom"], probs)
    print(f"+{h}d: {train['bloom'].size:,} training rows, {valid['bloom'].size:,} validation rows, "
          f"{(perf_counter() - started) / 60:.1f} min")
    del train, valid, classifier, regressor
screening = pd.DataFrame(screen_rows).set_index(["Horizon", "Method"])
display(screening.style.format({"Brier score": "{:.5f}", "Skill vs usual frequency (%)": "{:+.1f}", "95% low": "{:+.1f}",
                                "95% high": "{:+.1f}", "Skill vs anomaly rule (%)": "{:+.1f}", "AUC": "{:.3f}"}))

# The candidate for the final test, by a rule fixed in advance: the lowest Brier
# score on 2024, averaged over the three lead times.
CANDIDATES = ["Classifier", "Classifier + area state", "Model in use, as a probability"]
BEST = screening.loc[(slice(None), CANDIDATES), "Brier score"].groupby("Method").mean().idxmin()
print(f"Best candidate on 2024: {BEST}")
display(pd.DataFrame([{"Horizon": f"+{h}d", "Method": m, "Warning threshold": thresholds[(h, m)]}
                      for h in horizons for m in ["Anomaly rule", BEST]]).set_index(["Horizon", "Method"]))

+1d: 5,425,586 training rows, 2,506,143 validation rows, 0.9 min
+4d: 5,423,327 training rows, 2,513,473 validation rows, 0.9 min
+7d: 5,425,316 training rows, 2,540,205 validation rows, 0.9 min


Best candidate on 2024: Classifier


Warning threshold
Horizon Method                         
+1d     Anomaly rule               0.12
        Classifier                 0.19
+4d     Anomaly rule               0.13
        Classifier                 0.18
+7d     Anomaly rule               0.09
        Classifier                 0.15

In [5]:
# The final test, once: refit on 2017–2024, score on 2025.
final, test_rows, warning_rows, reliability_rows, breakdown_rows = {}, [], [], [], []
for h in horizons:
    started = perf_counter()
    fit_rows = rows("train+validation", h, FINAL_SAMPLE_FRACTION)
    rules = fit_rules(fit_rows)
    if BEST.startswith("Classifier"):
        names = FEATURE_SETS["18 inputs + area state" if "area" in BEST else "18 inputs"]
        model = HistGradientBoostingClassifier(loss="log_loss", **TREE_PARAMS).fit(fit_rows["X"][names], fit_rows["bloom"])
        curve = None
    else:
        names = FEATURES
        model = in_use["models"][h]          # trained on 2017–2024 already
        curve = calibrate_regression(model, fit_rows)
    final[h] = {"model": model, "curve": curve, "rules": rules}
    del fit_rows

    test = rows("test", h, 1.0)
    probs = apply_rules(rules, test)
    probs["Bloom model"] = (model.predict_proba(test["X"][names])[:, 1] if curve is None
                            else regression_probability(model, curve, test))
    for name, p in probs.items():
        skill, low, high = blooms.brier_skill_interval(test["bloom"], p, probs["Usual frequency"], test["t"])
        vs_rule, vs_low, vs_high = blooms.brier_skill_interval(test["bloom"], p, probs["Anomaly rule"], test["t"])
        test_rows.append({"Horizon": f"+{h}d", "Method": name, "Brier score": blooms.brier(p, test["bloom"]),
                          "Skill vs usual frequency (%)": skill, "95% low": low, "95% high": high,
                          "Skill vs anomaly rule (%)": vs_rule, "vs rule 95% low": vs_low, "vs rule 95% high": vs_high,
                          "AUC": roc_auc_score(test["bloom"], p)})
    reliability_rows += [{"Horizon": f"+{h}d", "From": lo, "To": hi, "Forecasts": n, "Mean probability": mp, "Observed frequency": of}
                         for lo, hi, n, mp, of in blooms.reliability(probs["Bloom model"], test["bloom"])]

    # Yes/no warnings, with the thresholds chosen on 2024; and the model in use's own value
    # forecast as a warning (a bloom forecast when it predicts at least 1.6 × the usual value).
    for name, key in [("Bloom model", BEST), ("Anomaly rule", "Anomaly rule")]:
        warning_rows.append({"Horizon": f"+{h}d", "Method": name, **blooms.warning_scores(probs[name], test["bloom"], thresholds[(h, key)])})
    value_forecast = in_use["models"][h].predict(test["X"][FEATURES]) - test["clim_log"] >= config.BLOOM_ANOMALY_LOG10
    warning_rows.append({"Horizon": f"+{h}d", "Method": "Model in use, value ≥ 1.6 × usual",
                         **blooms.warning_scores(value_forecast.astype(float), test["bloom"], 0.5)})

    seasons = np.array([SEASONS[m] for m in test["month"]])
    near = coast_km[test["i"], test["j"]] <= COAST_KM
    for group, sel in [*[(s, seasons == s) for s in ["Winter", "Spring", "Summer", "Autumn"]],
                       (f"Within {COAST_KM} km of the coast", near), ("Further offshore", ~near)]:
        b = test["bloom"][sel]
        breakdown_rows.append({"Horizon": f"+{h}d", "Group": group, "Pixel-days": int(sel.sum()), "Blooms": int(b.sum()),
                               "Skill vs usual frequency (%)": 100 * (1 - blooms.brier(probs["Bloom model"][sel], b) / blooms.brier(probs["Usual frequency"][sel], b)),
                               "Skill vs anomaly rule (%)": 100 * (1 - blooms.brier(probs["Bloom model"][sel], b) / blooms.brier(probs["Anomaly rule"][sel], b))})
    print(f"+{h}d: fitted and tested in {(perf_counter() - started) / 60:.1f} min")
    del test

display(pd.DataFrame(test_rows).set_index(["Horizon", "Method"]).style.format(
    {"Brier score": "{:.5f}", "Skill vs usual frequency (%)": "{:+.1f}", "95% low": "{:+.1f}", "95% high": "{:+.1f}",
     "Skill vs anomaly rule (%)": "{:+.1f}", "vs rule 95% low": "{:+.1f}", "vs rule 95% high": "{:+.1f}", "AUC": "{:.3f}"}))
display(pd.DataFrame(warning_rows).set_index(["Horizon", "Method"]).style.format(
    {"Threshold": "{:.2f}", "Warnings": "{:,.0f}", "Blooms": "{:,.0f}", "Hit rate (%)": "{:.1f}",
     "False alarm ratio (%)": "{:.1f}", "Critical success index (%)": "{:.1f}"}))
display(pd.DataFrame(breakdown_rows).set_index(["Horizon", "Group"]).style.format(
    {"Pixel-days": "{:,.0f}", "Blooms": "{:,.0f}", "Skill vs usual frequency (%)": "{:+.1f}", "Skill vs anomaly rule (%)": "{:+.1f}"}))
display(pd.DataFrame(reliability_rows).set_index(["Horizon", "From", "To"]).style.format(
    {"Forecasts": "{:,.0f}", "Mean probability": "{:.3f}", "Observed frequency": "{:.3f}"}))

+1d: fitted and tested in 0.8 min
+4d: fitted and tested in 0.7 min
+7d: fitted and tested in 0.7 min


In [6]:
# Save the bloom model and its results. Not used by the live forecast (step 07) yet.
from datetime import datetime, timezone
joblib.dump({
    "method": BEST,
    "models": {h: final[h]["model"] for h in horizons} if BEST.startswith("Classifier") else None,
    "regression_model_file": None if BEST.startswith("Classifier") else config.MODEL_PATH.name,
    "calibration_curves": None if BEST.startswith("Classifier") else {h: final[h]["curve"] for h in horizons},
    "feature_names": names if BEST.startswith("Classifier") else FEATURES,
    "bloom_anomaly_log10": config.BLOOM_ANOMALY_LOG10,
    "usual_frequency": {h: final[h]["rules"]["base"].reshape(12, *mask.shape) for h in horizons},
    "anomaly_rule": {h: final[h]["rules"]["anomaly"] for h in horizons},
    "anomaly_bins": ANOMALY_BINS,
    "warning_thresholds": {f"+{h}d {m}": t for (h, m), t in thresholds.items()},
    "tree_settings": TREE_PARAMS,
    "bloom_counts": counts,
    "screening": screening.reset_index().to_dict("records"),
    "test_scores": test_rows,
    "warnings": warning_rows,
    "breakdown": breakdown_rows,
    "reliability": reliability_rows,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}, config.BLOOM_MODEL_PATH, compress=3)
print(f"Saved: {config.BLOOM_MODEL_PATH}")

Saved: /Users/alangatt/git/mcast.pages.chl_prediction/experiment_v2/models/bloom_step10.joblib
